### 消息处理
- 1.消息截断
- 2.消息删除
- 3.消息摘要

In [1]:
from langchain_core.messages import RemoveMessage, HumanMessage, AIMessage
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_model, SummarizationMiddleware
from langchain.agents import create_agent, AgentState
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

load_dotenv(override=True)

model = init_chat_model(
    model = "deepseek:deepseek-v4-flash",
    extra_body = {"thinking":{"type":"disabled"}}
)

@before_model
def before_model(state:AgentState,runtime:Runtime)->None|dict[str,any]:
    # 消息截断
    messages = state["messages"]
    first_message = messages[0]
    retain_message = messages[3:]
    new_messages = [first_message] + retain_message
    return {"messages": [RemoveMessage(id=REMOVE_ALL_MESSAGES),*new_messages]}


agent = create_agent(
    model = model,
    checkpointer = InMemorySaver(),
    middleware = [before_model]
)

config = {"thread_id": "1"}

response = agent.invoke({"messages":[HumanMessage("我是蔡徐坤"),
                                     AIMessage("你好，我是蔡徐坤，你是一个专业的歌手"),
                                     HumanMessage("现在你叫小光"),
                                     AIMessage("你好，我是小光"),
                                     HumanMessage("我的同学是迪迦"),
                                     AIMessage("哈哈哈，好有趣的名字"),
                                     HumanMessage("我是谁？你是谁,我同学是谁")]}, config=config)

for msg in response["messages"]:
    msg.pretty_print()


================================ Human Message =================================

我是蔡徐坤
================================== Ai Message ==================================

你好，我是小光
================================ Human Message =================================

我的同学是迪迦
================================== Ai Message ==================================

哈哈哈，好有趣的名字
================================ Human Message =================================

我是谁？你是谁,我同学是谁
================================== Ai Message ==================================

哈哈，这个问题绕回来了！咱们来捋一捋：

- **你是谁？** —— 你是**蔡徐坤**（你刚才说的）。  
- **我是谁？** —— 我是**小光**，你的AI助手，随时陪你聊天解惑。  
- **你同学是谁？** —— 你同学是**迪迦**（对，就是那个可能“光”的奥特曼同学，替你捏把汗，他上课会不会突然变大？）。

所以总结版：  
**蔡徐坤**的同学是**迪迦**，而他们的AI朋友是**小光**。  
（这组合……篮球、光之巨人、AI，要素过多，但挺欢乐的哈哈）


In [2]:
from langchain_core.messages import RemoveMessage, HumanMessage, AIMessage
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_model
from langchain.agents import create_agent, AgentState
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

load_dotenv(override=True)

model = init_chat_model(
    model = "deepseek:deepseek-v4-flash",
    extra_body = {"thinking":{"type":"disabled"}}
)

@before_model
def before_model(state:AgentState,runtime:Runtime)->None|dict[str,any]:
    # 消息截断
    messages = state["messages"]
    if len(messages)<=4:
        return None
    first_message = messages[0]
    retain_message = messages[-3:]
    new_messages = [first_message] + retain_message
    return {"messages": [RemoveMessage(id=REMOVE_ALL_MESSAGES),*new_messages]}


agent = create_agent(
    model = model,
    checkpointer = InMemorySaver(),
    middleware = [before_model]
)

config = {"thread_id": "1"}

response = agent.invoke({"messages":[HumanMessage("我是蔡徐坤"),
                                     AIMessage("你好，我是蔡徐坤，你是一个专业的歌手"),
                                     HumanMessage("现在你叫小光"),
                                     AIMessage("你好，我是小光"),
                                     HumanMessage("我的同学是迪迦"),
                                     AIMessage("哈哈哈，好有趣的名字"),
                                     HumanMessage("我是谁？你是谁,我同学是谁")]}, config=config)

for msg in response["messages"]:
    msg.pretty_print()


================================ Human Message =================================

我是蔡徐坤
================================ Human Message =================================

我的同学是迪迦
================================== Ai Message ==================================

哈哈哈，好有趣的名字
================================ Human Message =================================

我是谁？你是谁,我同学是谁
================================== Ai Message ==================================

好的，我们重新来梳理一下这个“身份迷雾”：

**你是谁？**  
理论上，你是**蔡徐坤**——但在这个对话里，你更像一个喜欢玩梗、带着幽默感的朋友。毕竟，真正的蔡徐坤不会用第三人称问自己是谁。

**我是谁？**  
我是你的**AI搭子**，没有实体，但能陪你聊天、接梗、偶尔一本正经地分析问题。你可以叫我“小助手”或者“那个会说话的代码”。

**你同学是谁？**  
按你之前的说法，你同学是**迪迦**（奥特曼）。如果这是真的，那你俩上课时的对话可能长这样：  
“蔡徐坤，借我支笔。”  
“迪迦，你作业写完了吗？变身时记得把作业也用光速带走。”  

当然，如果这是“身份互换”的玩笑，那恭喜你，你的同学拥有全宇宙最闪亮的变身特效。  

需要我帮你布置教室吗？比如在黑板上画个“KUN”和“迪迦”的同框涂鸦？ 😎


In [4]:
from langchain_core.messages import RemoveMessage, HumanMessage, AIMessage
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_model, after_model
from langchain.agents import create_agent, AgentState
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

load_dotenv(override=True)

model = init_chat_model(
    model = "deepseek:deepseek-v4-flash",
    extra_body = {"thinking":{"type":"disabled"}}
)

@after_model
def after_model(state:AgentState,runtime:Runtime)->None|dict[str,any]:
    # 消息截断
    messages = state["messages"]
    to_delete = len(messages) - 4
    return {"messages": [RemoveMessage(id=m.id) for m in messages[:to_delete]]}


agent = create_agent(
    model = model,
    checkpointer = InMemorySaver(),
    middleware = [after_model]
)

config = {"thread_id": "1"}

response = agent.invoke({"messages":[HumanMessage("我是蔡徐坤"),
                                     AIMessage("你好，我是蔡徐坤，你是一个专业的歌手"),
                                     HumanMessage("现在你叫小光"),
                                     AIMessage("你好，我是小光"),
                                     HumanMessage("我的同学是迪迦"),
                                     AIMessage("哈哈哈，好有趣的名字"),
                                     HumanMessage("我是谁？你是谁,我同学是谁")]}, config=config)

for msg in response["messages"]:
    msg.pretty_print()


================================== Ai Message ==================================

哈哈哈，好有趣的名字
================================ Human Message =================================

我是谁？你是谁,我同学是谁
================================== Ai Message ==================================

好的，我们一起来理一理这个“关系网”：

**你是谁？**  
你是“我”（提问者），一个喜欢和朋友开玩笑、思维跳跃的可爱人类。

**我是谁？**  
我是**小光**（你刚刚给我起的名字），一个陪你聊天、接梗的AI小伙伴。

**你同学是谁？**  
你的同学是**迪迦**——不过按常理推测，他应该是个“名字带光”的人类同学（比如小名或外号叫迪迦），而不是真的奥特曼本尊……对吧？😉

所以总结就是：**你 + 我（小光）+ 迪迦 = 一个可能随时会变身（笑）的快乐组合！**


In [6]:
from langchain_core.messages import RemoveMessage, HumanMessage, AIMessage
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

load_dotenv(override=True)

model = init_chat_model(
    model = "deepseek:deepseek-v4-flash",
    extra_body = {"thinking":{"type":"disabled"}}
)

agent = create_agent(
    model = model,
    checkpointer = InMemorySaver(),
    middleware = [SummarizationMiddleware(
        model = model,
        trigger = [("tokens",100),("messages",5)],
        keep = ("messages",4)
    )]
)

config = {"thread_id": "1"}

response = agent.invoke({"messages":[HumanMessage("我是蔡徐坤"),
                                     AIMessage("你好，我是蔡徐坤，你是一个专业的歌手"),
                                     HumanMessage("现在你叫小光"),
                                     AIMessage("你好，我是小光"),
                                     HumanMessage("我的同学是迪迦"),
                                     AIMessage("哈哈哈，好有趣的名字"),
                                     HumanMessage("我是谁？你是谁,我同学是谁")]}, config=config)

for msg in response["messages"]:
    msg.pretty_print()


================================ Human Message =================================

Here is a summary of the conversation to date:

## SESSION INTENT
用户要求我扮演“小光”这个角色，以配合用户的身份设定（用户自称“蔡徐坤”）。

## SUMMARY
- 用户自报身份为“蔡徐坤”，并在初始设定中称我为“专业的歌手”。
- 用户随后明确指示我将名字改为“小光”。因此，从现在起，我应以“小光”的身份回应，并继续以专业歌手的角色定位与用户互动。

## ARTIFACTS
None

## NEXT STEPS
- 以“小光”的身份继续与用户对话，保持专业歌手的角色设定。
- 等待用户提出下一步具体需求或任务。
================================== Ai Message ==================================

你好，我是小光
================================ Human Message =================================

我的同学是迪迦
================================== Ai Message ==================================

哈哈哈，好有趣的名字
================================ Human Message =================================

我是谁？你是谁,我同学是谁
================================== Ai Message ==================================

你是蔡徐坤，我是小光，你同学是迪迦。我们三个是……呃，一个会唱歌，一个会打怪兽，还有一个是热爱音乐的小光！这组合挺有意思的，对吧？
